In [1]:
# =====================================================
# SECTION 1: IMPORT LIBRARIES
# =====================================================

import pandas as pd
import os

In [2]:
# =====================================================
# SECTION 2: LOAD SMS DATASET
# =====================================================

sms_df = pd.read_csv(
    "../data/raw/sms/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "text"]
)

print("SMS dataset shape:", sms_df.shape)
print(sms_df.head())

SMS dataset shape: (5572, 2)
  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [3]:
# =====================================================
# SECTION 3: CLEAN SMS DATASET
# =====================================================

# Convert labels
sms_df["label"] = sms_df["label"].map({
    "ham": 0,
    "spam": 1
})

# Add source
sms_df["source"] = "sms"

# Clean
sms_df = sms_df.dropna().drop_duplicates()
sms_df["text"] = sms_df["text"].str.strip()

# 🔴 IMPORTANT CHECK
print("\nSMS source check:")
print(sms_df["source"].value_counts())


SMS source check:
source
sms    5169
Name: count, dtype: int64


In [4]:
# =====================================================
# SECTION 4: LOAD EMAIL DATASET
# =====================================================

email_df = pd.read_csv(
    "../data/raw/enron_email/spam_ham_dataset.csv"
)

print("Original columns:", email_df.columns)
print(email_df.head())

Original columns: Index(['Unnamed: 0', 'label', 'text', 'label_num'], dtype='object')
   Unnamed: 0 label                                               text  \
0         605   ham  Subject: enron methanol ; meter # : 988291\r\n...   
1        2349   ham  Subject: hpl nom for january 9 , 2001\r\n( see...   
2        3624   ham  Subject: neon retreat\r\nho ho ho , we ' re ar...   
3        4685  spam  Subject: photoshop , windows , office . cheap ...   
4        2030   ham  Subject: re : indian springs\r\nthis deal is t...   

   label_num  
0          0  
1          0  
2          0  
3          1  
4          0  


In [5]:
# =====================================================
# SECTION 5: CLEAN EMAIL DATASET
# =====================================================

# Remove duplicate columns
email_df = email_df.loc[:, ~email_df.columns.duplicated()]

# Keep required columns
email_df = email_df[["text", "label_num"]]

# Rename label
email_df = email_df.rename(columns={"label_num": "label"})

# Add source
email_df["source"] = "email"

# Clean
email_df = email_df.dropna().drop_duplicates()
email_df["text"] = email_df["text"].str.strip()

# 🔴 IMPORTANT CHECK
print("\nEmail source check:")
print(email_df["source"].value_counts())


Email source check:
source
email    4993
Name: count, dtype: int64


In [8]:
# =====================================================
# SECTION 6: VERIFY BOTH DATASETS BEFORE MERGING
# =====================================================

print("\nSMS shape:", sms_df.shape)
print("Email shape:", email_df.shape)

print("\nSMS sample:\n", sms_df.head(2))
print("\nEmail sample:\n", email_df.head(2))


SMS shape: (5169, 3)
Email shape: (4993, 3)

SMS sample:
    label                                               text source
0      0  Go until jurong point, crazy.. Available only ...    sms
1      0                      Ok lar... Joking wif u oni...    sms

Email sample:
                                                 text  label source
0  Subject: enron methanol ; meter # : 988291\r\n...      0  email
1  Subject: hpl nom for january 9 , 2001\r\n( see...      0  email


In [9]:
# =====================================================
# SECTION 7: COMBINE DATASETS
# =====================================================

combined_df = pd.concat([sms_df, email_df], ignore_index=True)

print("\nCombined dataset shape:", combined_df.shape)


Combined dataset shape: (10162, 3)


In [10]:
# =====================================================
# SECTION 8: CRITICAL VERIFICATION (DO NOT SKIP)
# =====================================================

print("\nSource distribution:")
print(combined_df["source"].value_counts())

print("\nLabel distribution:")
print(combined_df["label"].value_counts())


Source distribution:
source
sms      5169
email    4993
Name: count, dtype: int64

Label distribution:
label
0    8047
1    2115
Name: count, dtype: int64


In [11]:
# =====================================================
# SECTION 9: FINAL CLEANING + TYPE FIX
# =====================================================

combined_df["label"] = combined_df["label"].astype(int)

print("\nData types:\n", combined_df.dtypes)


Data types:
 label      int64
text      object
source    object
dtype: object


In [13]:
# =====================================================
# SECTION 10: SAVE FINAL DATASET
# =====================================================

combined_df.to_csv(
    "../data/processed/combined_spam_dataset.csv",
    index=False
)

print("\nDataset saved successfully!")


Dataset saved successfully!
